In [7]:
import os
import numpy as np
import pandas as pd
import argparse


In [8]:
'''
# Define arguments 
parser = argparse.ArgumentParser(
                    prog='adjusts_esp',
                    description='Ajuste de pronosticos hidrológicos usando tendencia climática de INUMET',
                    epilog='Jose Valles, DINAGUA, october2026')

# 
parser.add_argument('codcuenca_n2', help='brinda el codigo de cuenca nivel 2')
parser.add_argument('trimestre_tendencia', help='brinda el trimestre de la tendencia climática en formato acronimo de meses')
parser.add_argument('tercil_superior', help='brinda el valor del tercil superior')
parser.add_argument('tercil_medio', help='brinda el valor del tercil medio')
parser.add_argument('tercil_inferior', help='brinda el valor del tercil inferior')

args = parser.parse_args()
'''
args = argparse.Namespace(
    codcuenca_n2 ='62',
    trimestre_tendencia = "OND",
    tercil_superior = 0.40,
    tercil_medio = 0.30,
    tercil_inferior = 0.30,
)

In [9]:
TENDENCIA_CLIMATICA = {'superior': args.tercil_superior,
                       'medio': args.tercil_medio,
                       'inferior': args.tercil_inferior}

CODCUENCA_N2 = args.codcuenca_n2
TRIMESTRE_TENDENCIA = args.trimestre_tendencia

In [10]:
def process_precipitation(
    codcuenca: str,
    filename: str,
    start_year: int,
    end_year: int,
    precip_folder: str,
    accumulation_months: int = 3,
    alphap: float = 0.4,
    betap: float = 0.4,
    percentiles: list[float] = None,
    categoria_precip: list[str] = None,
    precip_cat: list[int] = None,
) -> pd.DataFrame:
    """Procesa series mensuales de precipitación por cuenca, calcula el acumulado

    hacia adelante y asigna categorías por cuantiles usando la fórmula de Cunnane.
    """
    if percentiles is None:
        percentiles = [0.333, 0.666]
    if categoria_precip is None:
        categoria_precip = ["superior", "medio", "inferior"]
    if precip_cat is None:
        precip_cat = [3, 2, 1]

    file_path = os.path.join(precip_folder, filename)
    precip_df = pd.read_csv(file_path).rename(
        columns={"-1": "Year", "-1.1": "Month"}
    )

    # Filtrado temporal extendido para permitir acumulación inicial
    precip_df = precip_df[
        (precip_df["Year"] >= start_year - 1) & (precip_df["Year"] <= end_year)
    ]

    cuenca_cols = [
        col for col in precip_df.columns if col.startswith(str(codcuenca))
    ]
    precip_df = precip_df[["Year", "Month"]].assign(
        Precip=precip_df[cuenca_cols].mean(axis=1)
    )
    precip_df["Date"] = pd.to_datetime(
        precip_df["Year"].astype(str) + "-" + precip_df["Month"].astype(str),
        format="%Y-%m",
    )

    # Acumulación hacia adelante
    acc_col = f"Precip_{accumulation_months}mo"
    precip_df[acc_col] = (
        precip_df["Precip"]
        .rolling(window=accumulation_months, min_periods=accumulation_months)
        .sum()
        .shift(-(accumulation_months - 1))
    )

    precip_df["startMonth"] = precip_df["Date"].dt.month
    precip_df["endMonth"] = precip_df["Date"].dt.month + (
        accumulation_months - 1
    )
    precip_df["startMonth"] = precip_df["startMonth"].apply(
        lambda x: x if x > 0 else x + 12
    )
    precip_df["endMonth"] = precip_df["endMonth"].apply(
        lambda x: x if x <= 12 else x - 12
    )

    precip_df = precip_df[
        (precip_df["Year"] >= start_year) & (precip_df["Year"] <= end_year)
    ].copy()

    month_letters = [
        "J",
        "F",
        "M",
        "A",
        "M",
        "J",
        "J",
        "A",
        "S",
        "O",
        "N",
        "D",
    ]

    def get_month_code(start, end):
        code = ""
        current = start
        while True:
            code += month_letters[current - 1]
            if current == end:
                break
            current = current % 12 + 1
        return code

    precip_df["month_code"] = precip_df.apply(
        lambda row: get_month_code(row["startMonth"], row["endMonth"]), axis=1
    )

    precip_df["rank_average"] = precip_df.groupby("month_code")[acc_col].rank()
    precip_df["complete%"] = precip_df.groupby("month_code")[acc_col].transform(
        lambda x: x.notnull().sum()
    )
    precip_df["pct_rank"] = (precip_df["rank_average"] - alphap) / (
        precip_df["complete%"] + 1 - alphap - betap
    )

    wr = precip_df["pct_rank"]
    criteria = [
        (wr >= percentiles[1]) & (wr <= 1.00),
        (wr >= percentiles[0]) & (wr < percentiles[1]),
        (wr >= 0.00) & (wr < percentiles[0]),
    ]
    precip_df["percentile_range"] = np.select(
        criteria, categoria_precip, default=None
    )
    precip_df["precipcat"] = np.select(criteria, precip_cat, default=np.nan)

    return precip_df


In [11]:
def ajustar_probabilidades_esp(
    codcuenca_n2: str,
    month_code: str,
    tendencia_climatica: dict[str, float],
    precip_esp_folder: str = "D:/Documentos/MATLAB/Modelación Temez DINAGUA/Calculos/esp_precip",
    years_cat_folder: str = "d:/GitHub/Sala_Situacion_Dinagua/Status_Outlook_Bulletin/waterbalance/output_csv/years_cat/",
    filename_precip: str = "Precip2025.csv",
    start_year: int = 1980,
    end_year: int = 2025,
) -> dict[str, float]:
    """Ajusta las probabilidades condicionales del ensamble ESP según los terciles

    de tendencia climática de INUMET.

    Parámetros:
    -----------
    codcuenca_n2 : str
        Código identificador de la cuenca Nivel 2 (ej. "62").
    month_code : str
        Código de 3 letras del trimestre objetivo (ej. "OND").
    tendencia_climatica : dict
        Diccionario con probabilidades de INUMET.
        Ej: {'superior': 0.40, 'medio': 0.30, 'inferior': 0.30}
    precip_esp_folder : str
        Directorio donde reside el archivo de precipitación histórica.
    years_cat_folder : str
        Directorio donde residen los CSV de salida del ESP continuo por cuenca.
    filename_precip : str
        Nombre del archivo CSV de precipitaciones.
    start_year : int
        Año de inicio de la serie histórica.
    end_year : int
        Año final de la serie histórica.

    Retorna:
    --------
    dict[str, float]
        Probabilidades condicionadas normalizadas para {'superior', 'normal', 'inferior'}.
    """
    # 1. Procesar series históricas de precipitación
    df_precip = process_precipitation(
        codcuenca=codcuenca_n2,
        filename=filename_precip,
        start_year=start_year,
        end_year=end_year,
        precip_folder=precip_esp_folder,
        accumulation_months=3,
        alphap=0.4,
        betap=0.4,
    )

    # 2. Filtrar por el trimestre de interés
    df_trimestre = df_precip[df_precip["month_code"] == month_code].copy()

    if df_trimestre.empty:
        raise ValueError(
            f"No se encontraron datos para el trimestre '{month_code}'."
        )

    # 3. Conteo de frecuencias observadas por tercil
    pct_count = df_trimestre["percentile_range"].value_counts()

    # 4. Cálculo de factores de ponderación climática (w_j = P_j / N_j)
    weights = {
        cat: (tendencia_climatica[cat] / pct_count[cat])
        if pct_count.get(cat, 0) > 0
        else 0.0
        for cat in ["superior", "medio", "inferior"]
    }

    # 5. Cargar salidas del ESP tradicional
    esp_file = os.path.join(years_cat_folder, f"{codcuenca_n2}.csv")
    if not os.path.exists(esp_file):
        raise FileNotFoundError(
            f"No se encontró el archivo del ESP tradicional: {esp_file}"
        )

    esp_traditional = pd.read_csv(esp_file, index_col=0)
    category_map = {
        "Above Normal": "superior",
        "Normal Range": "normal",
        "Below Normal": "inferior",
    }
    esp_traditional["category"] = (
        esp_traditional["category"]
        .map(category_map)
        .fillna(esp_traditional["category"])
    )

    # Añadir categoría hidrológica al dataframe del trimestre
    df_trimestre["esp_traditional"] = esp_traditional["category"].values

    # 6. Construir matriz de contingencia
    contingencia = (
        pd.crosstab(
            df_trimestre["esp_traditional"], df_trimestre["percentile_range"]
        )
        .reindex(
            index=["superior", "normal", "inferior"],
            columns=["superior", "medio", "inferior"],
            fill_value=0,
        )
        .to_dict(orient="index")
    )

    # 7. Computar probabilidades condicionales
    q_prob = {}
    for cat_q in ["superior", "normal", "inferior"]:
        q_prob[cat_q] = sum(
            contingencia[cat_q][cat_p] * weights[cat_p]
            for cat_p in ["superior", "medio", "inferior"]
        )

    return q_prob

In [12]:
probabilidades = ajustar_probabilidades_esp(
    codcuenca_n2 = CODCUENCA_N2,
    month_code = TRIMESTRE_TENDENCIA,
    tendencia_climatica=TENDENCIA_CLIMATICA
)

print(f"Superior: {probabilidades['superior'] * 100:.2f}%")
print(f"Normal:   {probabilidades['normal'] * 100:.2f}%")
print(f"Inferior: {probabilidades['inferior'] * 100:.2f}%")

Superior: 16.00%
Normal:   35.25%
Inferior: 48.75%
